# Actual orders (matched)

This version is aligned with the Bullwhip / incoherence workflow and writes `actual_order.pkl`.

In [1]:

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make local helper modules importable
search_roots = [Path.cwd(), Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()]
for root in search_roots:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    if (root / "inventory_pipeline.py").exists():
        break

from inventory_pipeline import run_base_loop


def first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("None of these files exists:\n" + "\n".join(map(str, paths)))


def model_candidates(models_dir, stem, prefixes=("721", "")):
    names = []
    for prefix in prefixes:
        if prefix:
            names.append(os.path.join(models_dir, f"{prefix}{stem}"))
        names.append(os.path.join(models_dir, stem))
    # de-duplicate while preserving order
    return list(dict.fromkeys(names))


In [2]:

# ---------- configuration ----------
MODELS_DIR = r"E:/yjz/Extension for hts/JayCode/Models/"
PREFIXES = ("721", "")
LEAD_TIME = 1
HORIZON = 28

future_path = first_existing(model_candidates(MODELS_DIR, "future_28.pkl", PREFIXES))
test = pd.read_pickle(future_path).reset_index(drop=True)
truth = test["y"].to_numpy(dtype=float)

n_windows = len(truth) // HORIZON
if len(truth) % HORIZON != 0:
    raise ValueError(f"truth length {len(truth)} is not divisible by HORIZON={HORIZON}.")

print("future file:", future_path)
print("rows:", len(test), "| windows:", n_windows, "| lead time:", LEAD_TIME)
test.head()


future file: E:/yjz/Extension for hts/JayCode/Models/721future_28.pkl
rows: 1195208 | windows: 42686 | lead time: 1


,unique_id,ds,y
0,TOTAL/FOODS/FOODS_1/FOODS_1_001,1914,4
1,TOTAL/FOODS/FOODS_1/FOODS_1_001,1915,5
2,TOTAL/FOODS/FOODS_1/FOODS_1_001,1916,7
3,TOTAL/FOODS/FOODS_1/FOODS_1_001,1917,1
4,TOTAL/FOODS/FOODS_1/FOODS_1_001,1918,9


In [3]:

# Perfect-information benchmark used by the existing Bullwhip notebook:
# fcst = truth, residual = 0, then save the generated order as actual_order.pkl.
residual = np.zeros_like(truth, dtype=float)

actual_df = run_base_loop(
    fcst=truth,
    truth=truth,
    residual=residual,
    NAME="actual",
    gap1=HORIZON,
    gap2=HORIZON,
    L_=LEAD_TIME,
).reset_index(drop=True)

actual_df.head()


100%|███████████████████████████████████████████████████████████████████████████| 42686/42686 [01:36<00:00, 444.04it/s]


,name,true_demand,forecasts,sst_90,arrival_90,ot_90,wip_90,ip_90,net_90,backlog_90,...,cb_95,sst_99,arrival_99,ot_99,wip_99,ip_99,net_99,backlog_99,ch_99,cb_99
0,actual,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,actual,5.0,5.0,0.0,0.0,5.0,5.0,0.0,-5.0,5.0,...,95.0,0.0,0.0,5.0,5.0,0.0,-5.0,5.0,0.0,495.0
2,actual,7.0,7.0,0.0,5.0,7.0,7.0,0.0,-7.0,7.0,...,133.0,0.0,5.0,7.0,7.0,0.0,-7.0,7.0,0.0,693.0
3,actual,1.0,1.0,0.0,7.0,1.0,1.0,0.0,-1.0,1.0,...,19.0,0.0,7.0,1.0,1.0,0.0,-1.0,1.0,0.0,99.0
4,actual,9.0,9.0,0.0,1.0,9.0,9.0,0.0,-9.0,9.0,...,171.0,0.0,1.0,9.0,9.0,0.0,-9.0,9.0,0.0,891.0


In [4]:

actual_order = pd.DataFrame({"actual_order": actual_df["ot_90"].to_numpy(dtype=float)})
actual_order.to_pickle("actual_order.pkl")

preview = pd.concat([test[["unique_id", "ds", "y"]], actual_order], axis=1)
print("saved:", os.path.abspath("actual_order.pkl"))
preview.head()


saved: E:\yjz\Extension for hts\JayCode\modified_inventory_files\actual_order.pkl


,unique_id,ds,y,actual_order
0,TOTAL/FOODS/FOODS_1/FOODS_1_001,1914,4,0.0
1,TOTAL/FOODS/FOODS_1/FOODS_1_001,1915,5,5.0
2,TOTAL/FOODS/FOODS_1/FOODS_1_001,1916,7,7.0
3,TOTAL/FOODS/FOODS_1/FOODS_1_001,1917,1,1.0
4,TOTAL/FOODS/FOODS_1/FOODS_1_001,1918,9,9.0


In [5]:

# Optional single-series check
sample_uid = "TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001"
preview[preview["unique_id"] == sample_uid].head(28)


,unique_id,ds,y,actual_order
341488,TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001,1914,2,0.0
341489,TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001,1915,0,0.0
341490,TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001,1916,0,0.0
341491,TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001,1917,0,0.0
341492,TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001,1918,0,0.0
341493,TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001,1919,1,1.0
341494,TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001,1920,1,1.0
341495,TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001,1921,0,0.0
341496,TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001,1922,6,6.0
341497,TOTAL/CA/CA_1/FOODS/FOODS_1/FOODS_1_001,1923,1,1.0
